In [96]:
%reset -f
import pandas as pd
from jupyter_core.migrate import regex
import yfinance as yf
import matplotlib.pyplot as plt
import numpy as np

In [97]:
spx = yf.download("^GSPC", start="2026-02-17", end="2026-02-18", progress=False)
spot = spx["Close"].iloc[-1].item()
spot

/Users/euanbronsky/PyCharmMiscProject/.venv/lib/python3.14/site-packages/yfinance/scrapers/history.py:201: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()


6843.22021484375

In [98]:
spx_data = pd.read_csv('/Users/euanbronsky/Desktop/csv data spx.csv' ,
    sep=';')

spx_data['Name'] = spx_data['Name'].str.replace(r'^OPRA S&P 500 Index Option \d+\s+', '', regex = True)

spx_data['Expiry Date'] = pd.to_datetime(spx_data['Expiry Date'], format='%m/%d/%y')
valuation_date = pd.to_datetime('2026-02-17')

spx_data['TTM'] = (spx_data['Expiry Date'] - valuation_date).dt.days / 365
spx_data = spx_data[spx_data['TTM'] > 0]

cols = ['Bid Price', 'Ask Price', 'Mid Price', 'Open Interest', 'Strike Price']

spx_data[cols] = spx_data[cols].apply(
    lambda s: s.str.replace('.', '', regex=False)
              .str.replace(',', '.', regex=False)
              .astype(float))

spx_data = spx_data[
    (spx_data['Bid Price'] > 0) &
    (spx_data['Ask Price'] > spx_data['Bid Price']) &
    (spx_data['Mid Price'] > 0)
]

spx_data['Rel Spread'] = (spx_data['Ask Price'] - spx_data['Bid Price']) / spx_data['Mid Price']
spx_data = spx_data[spx_data['Rel Spread'] < 0.10]

spx_data['M'] = spot / spx_data['Strike Price']

spx_otm = spx_data[
    (spx_data['Put / Call'] == 'Call') &
    (spx_data['M'] >= 0.9) &
    (spx_data['M'] <= 1.1)
]



In [99]:
r = 0.05

discount = spx_otm['Strike Price'] * np.exp(-r * spx_otm['TTM'])

call_mask = (
    (spx_otm['Mid Price'] >= np.maximum(spot - discount, 0)) &
    (spx_otm['Mid Price'] <= spot)
)
spx_otm = spx_otm[call_mask]

In [100]:
from scipy.stats import norm
import numpy as np
from scipy.optimize import brentq

def BS_price(par, r, T, K, St):
    sig = par

    d1 = (np.log(St / K) + (r + 0.5*sig**2)*T) / (sig*np.sqrt(T))
    d2 = d1 - sig*np.sqrt(T)

    Ct_BS = St*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2)

    return Ct_BS

def implied_vol(C_mkt, r, T, K, St):
    f = lambda sig: BS_price(sig, r, T, K, St) - C_mkt
    return brentq(f, 1e-7, 5.0)

In [105]:
# pick one maturity (example: nearest)
T0 = spx_otm['Expiry Date'].min()
smile = spx_otm[spx_otm['Expiry Date'] == T0]

K = smile['Strike Price'].to_numpy()
C = smile['Mid Price'].to_numpy()
T = smile['TTM'].to_numpy()

iv = np.array([
    implied_vol(c, r, t, k, spot)
    for c, t, k in zip(C, T, K)
])
M = K / spot

# sort
idx = np.argsort(M)

plt.plot(M[idx], iv[idx])
plt.xlabel("log(K/S)")
plt.ylabel("Implied Volatility")
plt.title("SPX Volatility Smile")
plt.show()

In [102]:
Ct_mkt = spx_otm['Mid Price'].to_numpy()
Ct_strike = spx_otm['Strike Price'].to_numpy()
T_strike = spx_otm['TTM'].to_numpy()

#Pt_mkt = spx_otm.loc[spx_otm['Put / Call'] == 'Put', 'Mid Price']
#Pt_strike = spx_otm.loc[spx_otm['Put / Call'] == 'Put', 'Strike Price']

In [103]:
call_IV = [
    implied_vol(c, 0.05, t, k, spot)
    for c, t, k in zip(Ct_mkt, T_strike, Ct_strike)]

In [104]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib qt

M = np.asarray(spot / Ct_strike, dtype=float)
T = np.asarray(T_strike, dtype=float)
IV = np.asarray(call_IV, dtype=float)

mask = np.isfinite(M) & np.isfinite(T) & np.isfinite(IV)
M, T, IV = M[mask], T[mask], IV[mask]

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

surf = ax.plot_trisurf(T, M, IV, cmap='viridis', linewidth=0.2)

ax.set_xlabel('TTM')
ax.set_ylabel('Moneyness')
ax.set_zlabel('Implied Volatility')

fig.colorbar(surf, ax=ax, shrink=0.6)
plt.show()


In [44]:
import pandas as pd
import glob

files = glob.glob("/Users/euanbronsky/Downloads/spx_eod_2010-hixkhi/spx_eod*.txt")

df = pd.concat([pd.read_csv(f, sep=", ", engine = 'python') for f in files], ignore_index=True)

In [45]:
df = df.sort_values(["[QUOTE_DATE]", "[EXPIRE_DATE]", "[STRIKE]"])
df.columns = df.columns.str.replace(r'[\[\]]', '', regex=True)


In [46]:
df = df.reset_index(drop=True)


In [47]:
df

,QUOTE_UNIXTIME,QUOTE_READTIME,QUOTE_DATE,QUOTE_TIME_HOURS,UNDERLYING_LAST,EXPIRE_DATE,EXPIRE_UNIX,DTE,C_DELTA,C_GAMMA,...,P_LAST,P_DELTA,P_GAMMA,P_VEGA,P_THETA,P_RHO,P_IV,P_VOLUME,STRIKE_DISTANCE,STRIKE_DISTANCE_PCT
0,1262638800,2010-01-04 16:00,2010-01-04,16.0,1132.99,2010-01-07,1262898000,3.0,1.00000,0.00000,...,0.05,-0.00077,0.00004,0.00439,-0.02116,-0.00049,0.64013,550.0,208.0,0.184
1,1262638800,2010-01-04 16:00,2010-01-04,16.0,1132.99,2010-01-07,1262898000,3.0,1.00000,0.00000,...,0.05,-0.00203,0.00006,0.00911,-0.04056,0.00000,0.59859,NaN,183.0,0.162
2,1262638800,2010-01-04 16:00,2010-01-04,16.0,1132.99,2010-01-07,1262898000,3.0,1.00000,0.00000,...,0.10,-0.00369,0.00022,0.01302,-0.05898,-0.00071,0.53885,NaN,158.0,0.139
3,1262638800,2010-01-04 16:00,2010-01-04,16.0,1132.99,2010-01-07,1262898000,3.0,1.00000,0.00000,...,0.15,-0.00894,0.00038,0.02786,-0.12758,-0.00086,0.50273,NaN,133.0,0.117
4,1262638800,2010-01-04 16:00,2010-01-04,16.0,1132.99,2010-01-07,1262898000,3.0,1.00000,0.00000,...,0.20,-0.01251,0.00066,0.03857,-0.15692,-0.00147,0.42725,720.0,108.0,0.095
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
233243,1293829200,2010-12-31 16:00,2010-12-31,16.0,1258.02,2013-12-19,1387486800,1084.0,0.06019,0.00031,...,0.00,-0.96201,0.00000,0.00000,-0.04300,-56.34125,0.00034,NaN,642.0,0.510
233244,1293829200,2010-12-31 16:00,2010-12-31,16.0,1258.02,2013-12-19,1387486800,1084.0,0.04035,0.00028,...,754.52,-1.00000,0.00000,472.21645,0.00000,0.00000,-0.00034,0.0,742.0,0.590
233245,1293829200,2010-12-31 16:00,2010-12-31,16.0,1258.02,2013-12-19,1387486800,1084.0,0.01634,0.00009,...,990.94,-0.96231,0.00000,0.00000,-0.04260,-66.72045,-0.00003,0.0,992.0,0.789
233246,1293829200,2010-12-31 16:00,2010-12-31,16.0,1258.02,2013-12-19,1387486800,1084.0,0.00967,0.00004,...,1225.70,-1.00000,0.00000,470.73225,0.00000,0.00000,NaN,0.0,1242.0,0.987
